In [68]:
from utilities import *

df = load_file("./Data/SPY_5min_merged.csv", date_format="%Y-%m-%d %H:%M:%S")
print(df.dtypes)
print(df.head())
print(df.tail())

timestamp    datetime64[ns]
open                float64
high                float64
low                 float64
close               float64
volume                int64
dtype: object
            timestamp      open      high       low     close  volume
0 2023-03-01 04:00:00  385.2770  385.8490  384.4432  385.8490   17615
1 2023-03-01 04:05:00  385.8199  385.8199  385.5097  385.5679    7766
2 2023-03-01 04:10:00  385.6260  385.7036  385.6260  385.7036    1505
3 2023-03-01 04:15:00  385.7133  385.7230  385.5097  385.5194    4450
4 2023-03-01 04:20:00  385.5194  385.5485  385.1897  385.2479    4190
                timestamp      open      high       low     close   volume
96536 2025-02-28 19:40:00  592.3891  592.4489  592.2993  592.4190    28729
96537 2025-02-28 19:45:00  592.4190  592.4489  592.2694  592.3193     6240
96538 2025-02-28 19:50:00  592.3690  592.4190  592.3193  592.4190     8114
96539 2025-02-28 19:55:00  592.4090  592.5685  592.3492  592.5685    15501
96540 2025-02-28 20:00:

In [69]:
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['time_val'] = df['hour'] * 100 + df['minute']  # Creates time in HHMM format

df_regular_hours = df[(df['time_val'] >= 930) & (df['time_val'] <= 1555)]
df_regular_hours.drop(['hour', 'minute', 'time_val'], axis=1, inplace=True)

print(df.shape)
print(df_regular_hours.shape)

(96541, 9)
(39156, 6)


C:\Users\Reyes\AppData\Local\Temp\ipykernel_8288\1051571800.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [70]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# df_regular_hours = df_regular_hours.tail(2000)

initial_candles_shown = 100
initial_xrange = [df_regular_hours['timestamp'].iloc[-initial_candles_shown], df_regular_hours['timestamp'].iloc[-1] + pd.Timedelta(minutes=5)]

initial_yrange = [df_regular_hours.tail(initial_candles_shown)['low'].min() - 1,
                  df_regular_hours.tail(initial_candles_shown)['high'].max() + 1]

colors = {
    'bearish_wick': '#FF6347',
    'bearish_body': '#FF6347',
    'bearish_border': '#CC503A',
    
    'bullish_body': '#32CD32',
    'bullish_wick': '#32CD32',
    'bullish_border': '#28A028',
    
    'background': '#FAFAD2',
    'foreground': '#228B22',
    'text': '#228B22',
    'grid': '#D3D3D3'
}

fig = make_subplots()

fig.add_trace(
    go.Candlestick(
        x=df_regular_hours['timestamp'],
        open=df_regular_hours['open'],
        high=df_regular_hours['high'],
        low=df_regular_hours['low'],
        close=df_regular_hours['close'],
        name="Price",
        increasing={
            'line': {'color': colors['bullish_border'], 'width': 0.6},
            'fillcolor': colors['bullish_body']
        },
        
        decreasing={
            'line': {'color': colors['bearish_border'], 'width': 0.6},
            'fillcolor': colors['bearish_body']
        }
    )
)

fig.update_layout(
    paper_bgcolor=colors['background'],
    plot_bgcolor=colors['background'],
    font={'color': colors['text']},

    xaxis=dict(
        type='date',
        rangeslider=dict(visible=False), # cleaner
        fixedrange=False,  # Allow zooming on x-axis
        # autorange=True, # this must be disabled IF we set an initial range window
        range=initial_xrange,
        # Limit to last `initial_candles_shown` candles initially, plus margin on the right for aesthetics

        rangebreaks=[
            dict(bounds=["sat", "mon"]),  # Hide weekends
            dict(bounds=[17, 9], pattern="hour")  # Hide non-trading hours
        ],
        showgrid=False
    ),
    yaxis=dict(
        # autorange=True,  # Enable autorange for y-axis
        range=initial_yrange,
        fixedrange=False,  # VERY IMPORTANT TO OVERRIDE THE DEFAULT BEHAVIOR - allow y-axis zooming

        showgrid=True,
        gridcolor=colors['grid'],
        griddash='dot',
        gridwidth=1,
    ),
    margin=dict(l=20, r=20, t=40, b=20),
)

config = {
    'displayModeBar': True,
    'scrollZoom': False,
    'displaylogo': False
}

fig.show(config=config)

In [60]:
df = df_regular_hours

for day, group in df.groupby(df["timestamp"].dt.date):
    opening_range = group[group['timestamp'].dt.strftime('%H:%M').isin(['09:30', '09:35', '09:40'])]
    ORH = opening_range['high'].max()
    ORL = opening_range['low'].min()

    # for every following candle, wait for a break and close outside the OR
    # how do we quantify how significant the break is?

    break